# Chapitre 9 - Exercices 9.6 à 9.10

Ce notebook traite les exercices du chapitre 9 consacrés à l'ordre, aux horloges
logiques et au consensus :

- **9.6** : horloges de Lamport et vectorielles à la main (calcul) ;
- **9.7** : identifiants horodatés et décalage des horloges, une formule et sa
  vérification (calcul et mini-lab) ;
- **9.8** : taille et placement d'une grappe de consensus face aux pannes corrélées
  (calcul) ;
- **9.9** : casser Raft en supprimant la restriction d'élection (mini-lab) ;
- **9.10** : régler le délai d'élection (mini-lab).

In [1]:
import math
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import consistency as cs
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)

## Exercice 9.6 : horloges de Lamport et vectorielles à la main

**Énoncé.** Trois nœuds P, Q, R. Dans l'ordre du temps réel : P a un événement local
`p1` ; R a un événement local `r1` ; P envoie m1 à Q (`p2`) ; R envoie m3 à P (`r2`) ; Q
reçoit m1 (`q1`) ; P reçoit m3 (`p3`) ; Q envoie m2 à R (`q2`) ; Q a un événement local
`q3` ; R reçoit m2 (`r3`).

1. Calculer le compteur de Lamport et l'horloge vectorielle [P, Q, R] de chaque événement.
2. Pour les paires (`p3`, `q2`), (`r1`, `q3`) et (`p1`, `r3`), dire si elles sont
   causalement liées ou concurrentes, et comparer avec l'ordre de Lamport.
3. A-t-on « $L(a) < L(b) \Rightarrow a \to b$ » ? Et « $V(a) < V(b) \Rightarrow a \to b$ » ?

**Rappels.** Relation « s'est produit avant », horloges de Lamport et vectorielles (cours 2,
sections 1, 4 et 5 ; section *Lamport timestamps*).

**Solution.**

*1.* Lamport (incrément à chaque événement, rattrapage à la réception) : `p1` = 1, `r1` = 1,
`p2` = 2, `r2` = 2, `q1` = max(0, 2) + 1 = 3, `p3` = max(2, 2) + 1 = 3, `q2` = 4, `q3` = 5,
`r3` = max(2, 4) + 1 = 5. Vecteurs : `p1` [1,0,0], `r1` [0,0,1], `p2` [2,0,0], `r2` [0,0,2],
`q1` [2,1,0], `p3` [3,0,2], `q2` [2,2,0], `q3` [2,3,0], `r3` [2,2,3].

*2.* (`p3`, `q2`) : [3,0,2] et [2,2,0], chacun plus grand sur une composante :
**concurrents** ; Lamport les range pourtant (3, P) < (4, Q). (`r1`, `q3`) : [0,0,1] et
[2,3,0] : **concurrents** ; Lamport dit 1 < 5. (`p1`, `r3`) : [1,0,0] ≤ [2,2,3] : `p1` → `r3`
(chaîne p1, p2, m1, q1, q2, m2, r3), et Lamport est cohérent (1 < 5).

*3.* Non pour Lamport : la causalité implique l'ordre des compteurs, pas l'inverse
(contre-exemple `p3`, `q2`). Oui pour les vecteurs : $V(a) < V(b) \Leftrightarrow a \to b$.

In [2]:
nodes = ["P", "Q", "R"]
lam = {n: cs.LamportClock(n) for n in nodes}
vec = {n: cs.VectorClock(n, nodes) for n in nodes}
sent, ev = {}, {}
for kind, node, name, m in [("local", "P", "p1", None), ("local", "R", "r1", None), ("send", "P", "p2", "m1"),
                            ("send", "R", "r2", "m3"), ("recv", "Q", "q1", "m1"), ("recv", "P", "p3", "m3"),
                            ("send", "Q", "q2", "m2"), ("local", "Q", "q3", None), ("recv", "R", "r3", "m2")]:
    if kind == "recv":
        lam[node].observe(sent[m][0]); vec[node].observe(sent[m][1])
    ev[name] = (lam[node].tick(), vec[node].tick())
    if kind == "send":
        sent[m] = ev[name]
display(pd.DataFrame({k: dict(lamport=v[0][0], vecteur=list(v[1].values())) for k, v in ev.items()}).T)
pd.DataFrame([dict(paire=f"({a}, {b})", relation=cs.vc_compare(ev[a][1], ev[b][1]), lamport=f"{ev[a][0]} vs {ev[b][0]}")
              for a, b in [("p3", "q2"), ("r1", "q3"), ("p1", "r3")]])

,lamport,vecteur
p1,1,"[1, 0, 0]"
r1,1,"[0, 0, 1]"
p2,2,"[2, 0, 0]"
r2,2,"[0, 0, 2]"
q1,3,"[2, 1, 0]"
p3,3,"[3, 0, 2]"
q2,4,"[2, 2, 0]"
q3,5,"[2, 3, 0]"
r3,5,"[2, 2, 3]"


,paire,relation,lamport
0,"(p3, q2)",||,"(3, 'P') vs (4, 'Q')"
1,"(r1, q3)",||,"(1, 'R') vs (5, 'Q')"
2,"(p1, r3)",<,"(1, 'P') vs (5, 'R')"


Le code retrouve exactement les valeurs calculées à la main.

> **À retenir.** $a \to b \Rightarrow L(a) < L(b)$, mais pas la réciproque ; les horloges
> vectorielles caractérisent exactement la causalité.